In [1]:
# Install required libraries if they are missing
!pip install -q arxiv pymupdf

import argparse
import json
import re
import time
import urllib.request
from datetime import datetime
from pathlib import Path

import arxiv
import fitz  # PyMuPDF

# ---------------------------------------------------------------- settings
CATEGORY = "cs.CL"
DATE_FROM = "202501010000"   # submittedDate range (YYYYMMDDHHMM). Change if you like,
DATE_TO = "202610312359"     # but keep it fixed so the corpus stays reproducible.

# sub-topic label -> phrases searched in title OR abstract
TOPICS = {
    "machine translation": ["machine translation", "neural machine translation"],
    "question answering": ["question answering"],
    "text summarization": ["text summarization", "abstractive summarization",
                           "extractive summarization", "document summarization"],
    "dialogue systems": ["dialogue system", "dialogue systems", "task-oriented dialogue",
                         "conversational agent", "dialogue generation"],
}

# Length tiers from the assignment: short <6, medium 7-9, long >9 pages.
# A 6-page paper is not defined by the assignment. We put it in "medium".
# Decide this yourselves and state it in your report.
SHORT_MAX = 5
MEDIUM_MAX = 9

DATA = Path("data")
PDF_DIR = DATA / "pdfs"
TEXT_DIR = DATA / "texts"
CORPUS_FILE = DATA / "corpus.json"
LOG_FILE = DATA / "collection_log.json"


# ----------------------------------------------------------------- helpers
def build_query(phrases):
    terms = " OR ".join(f'ti:"{p}" OR abs:"{p}"' for p in phrases)
    return (f"cat:{CATEGORY} AND ({terms}) "
            f"AND submittedDate:[{DATE_FROM} TO {DATE_TO}]")


def length_tier(pages):
    if pages <= SHORT_MAX:
        return "short"
    if pages <= MEDIUM_MAX:
        return "medium"
    return "long"


def clean(text):
    text = text.replace("\x00", " ")
    return re.sub(r"[ \t]+", " ", re.sub(r"\n{3,}", "\n\n", text)).strip()


def save_corpus(papers):
    CORPUS_FILE.write_text(json.dumps(papers, indent=2, ensure_ascii=False),
                           encoding="utf-8")


def process_pdf(result, arxiv_id):
    """Download the PDF using urllib to prevent API mismatch errors, return (page_count, full_text)."""
    pdf_path = PDF_DIR / f"{arxiv_id}.pdf"
    if not pdf_path.exists():
        pdf_url = result.pdf_url
        urllib.request.urlretrieve(pdf_url, str(pdf_path))
        time.sleep(3)  # be polite to arXiv (they ask for ~3s between requests)
    with fitz.open(pdf_path) as doc:
        pages = len(doc)
        text = clean("\n".join(page.get_text() for page in doc))
    return pages, text


# -------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--per-topic", type=int, default=100)
    # Safely parse arguments in Jupyter notebooks using parse_known_args
    args, unknown = ap.parse_known_args()

    for d in (DATA, PDF_DIR, TEXT_DIR):
        d.mkdir(parents=True, exist_ok=True)

    papers = json.loads(CORPUS_FILE.read_text(encoding="utf-8")) if CORPUS_FILE.exists() else []
    seen = {p["arxiv_id"] for p in papers}

    client = arxiv.Client(page_size=100, delay_seconds=3, num_retries=5)
    log = {"category": CATEGORY, "date_from": DATE_FROM, "date_to": DATE_TO,
           "per_topic_target": args.per_topic, "run_started": datetime.now().isoformat(),
           "queries": {}}

    for topic, phrases in TOPICS.items():
        query = build_query(phrases)
        log["queries"][topic] = query
        have = sum(p["subtopic"] == topic for p in papers)
        print(f"\n=== {topic}: {have}/{args.per_topic} already collected ===")
        if have >= args.per_topic:
            continue

        search = arxiv.Search(query=query, max_results=args.per_topic * 4,
                              sort_by=arxiv.SortCriterion.Relevance)
        for r in client.results(search):
            if have >= args.per_topic:
                break
            if r.primary_category != CATEGORY:   # only genuine cs.CL papers
                continue
            arxiv_id = r.get_short_id().split("v")[0]
            if arxiv_id in seen:                  # a paper belongs to ONE sub-topic
                continue
            try:
                pages, text = process_pdf(r, arxiv_id)
            except Exception as e:
                print(f"  skip {arxiv_id}: {e}")
                continue
            if len(text) < 2000:                  # PDF extraction failed / scanned
                print(f"  skip {arxiv_id}: too little text")
                continue

            (TEXT_DIR / f"{arxiv_id}.txt").write_text(text, encoding="utf-8")
            papers.append({
                "title": " ".join(r.title.split()),
                "arxiv_id": arxiv_id,
                "url": f"https://arxiv.org/abs/{arxiv_id}",
                "published": r.published.date().isoformat(),
                "primary_category": r.primary_category,
                "abstract": " ".join(r.summary.split()),
                "subtopic": topic,
                "pages": pages,
                "length_category": length_tier(pages),
                "text_file": f"texts/{arxiv_id}.txt",
                "n_chars": len(text),
            })
            seen.add(arxiv_id)
            have += 1
            save_corpus(papers)                   # save after every paper (resumable)
            print(f"  [{have}/{args.per_topic}] {arxiv_id}  {pages}p  {papers[-1]['length_category']}")

    log["run_finished"] = datetime.now().isoformat()
    log["total_papers"] = len(papers)
    LOG_FILE.write_text(json.dumps(log, indent=2), encoding="utf-8")

    print("\n=== SUMMARY ===")
    print("total:", len(papers))
    for t in TOPICS:
        print(f"  {t}: {sum(p['subtopic'] == t for p in papers)}")
    for tier in ("short", "medium", "long"):
        print(f"  {tier}: {sum(p['length_category'] == tier for p in papers)}")


if __name__ == "__main__":
    main()

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(



=== machine translation: 0/100 already collected ===
  [1/100] 2503.03308  12p  long
  [2/100] 2603.27938  9p  medium
  [3/100] 2503.13837  7p  medium
  [4/100] 2504.21747  18p  long
  [5/100] 2501.12183  14p  long
  [6/100] 2605.09924  8p  medium
  [7/100] 2604.25702  7p  medium
  [8/100] 2502.01491  15p  long
  [9/100] 2512.09910  24p  long
  [10/100] 2501.02979  20p  long
  [11/100] 2512.15556  27p  long
  [12/100] 2607.19226  4p  short
  [13/100] 2502.01182  15p  long
  [14/100] 2610.32115  21p  long
  [15/100] 2610.11389  11p  long
  [16/100] 2608.12018  33p  long
  [17/100] 2608.03480  12p  long
  [18/100] 2504.17252  25p  long
  [19/100] 2505.14173  13p  long
  [20/100] 2606.25365  11p  long
  [21/100] 2502.04537  10p  long
  [22/100] 2501.01679  9p  medium
  [23/100] 2603.14873  8p  medium
  [24/100] 2603.17087  13p  long
  [25/100] 2605.20912  12p  long
  [26/100] 2608.08606  15p  long
  [27/100] 2503.07195  11p  long
  [28/100] 2509.16729  13p  long
  [29/100] 2503.06594  36

In [9]:
import json, collections
papers = json.load(open("data/corpus.json", encoding="utf-8"))
print(len(papers), len({p["arxiv_id"] for p in papers}))   # both should be 400
for t in sorted({p["subtopic"] for p in papers}):
    print(t, collections.Counter(p["length_category"] for p in papers if p["subtopic"] == t))

400 400
dialogue systems Counter({'long': 73, 'medium': 16, 'short': 11})
machine translation Counter({'long': 79, 'medium': 19, 'short': 2})
question answering Counter({'long': 78, 'medium': 18, 'short': 4})
text summarization Counter({'long': 69, 'medium': 22, 'short': 9})


In [10]:
!zip -r corpus.zip data -x "data/pdfs/*"
from google.colab import files; files.download("corpus.zip")

  adding: data/ (stored 0%)
  adding: data/texts/ (stored 0%)
  adding: data/texts/2510.08621.txt (deflated 62%)
  adding: data/texts/2506.10877.txt (deflated 60%)
  adding: data/texts/2606.15741.txt (deflated 66%)
  adding: data/texts/2604.03192.txt (deflated 60%)
  adding: data/texts/2502.08279.txt (deflated 64%)
  adding: data/texts/2501.04173.txt (deflated 63%)
  adding: data/texts/2604.07274.txt (deflated 69%)
  adding: data/texts/2610.32435.txt (deflated 62%)
  adding: data/texts/2606.18850.txt (deflated 62%)
  adding: data/texts/2501.14673.txt (deflated 63%)
  adding: data/texts/2610.19003.txt (deflated 63%)
  adding: data/texts/2506.08726.txt (deflated 72%)
  adding: data/texts/2505.14256.txt (deflated 61%)
  adding: data/texts/2511.06582.txt (deflated 71%)
  adding: data/texts/2502.14145.txt (deflated 59%)
  adding: data/texts/2503.23740.txt (deflated 60%)
  adding: data/texts/2504.09094.txt (deflated 64%)
  adding: data/texts/2509.11118.txt (deflated 64%)
  adding: data/texts

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>